# Loxodrómica vs Ortodrómica
Comparativa de rutas para navegación de altura (CY).

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/67_calculo_ortodromica_vs_loxodromica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

def calcular_distancias(lat1_deg, lon1_deg, lat2_deg, lon2_deg):
    # Convertir a radianes
    l1 = np.radians(lat1_deg)
    lo1 = np.radians(lon1_deg)
    l2 = np.radians(lat2_deg)
    lo2 = np.radians(lon2_deg)
    
    delta_lo = lo2 - lo1
    delta_l = l2 - l1
    
    # 1. ORTODRÓMICA (Círculo Máximo - Distancia más corta)
    # Fórmula de Haversine
    a = np.sin(delta_l/2)**2 + np.cos(l1) * np.cos(l2) * np.sin(delta_lo/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distancia_orto_nm = c * 3440.065 # Radio Tierra en NM aprox
    
    # 2. LOXODRÓMICA (Rumbo constante)
    # Latitud aumentada (Mercator)
    lat_aum1 = np.log(np.tan(np.pi/4 + l1/2))
    lat_aum2 = np.log(np.tan(np.pi/4 + l2/2))
    delta_lat_aum = lat_aum2 - lat_aum1
    
    if abs(delta_lat_aum) < 1e-6:
        # Navegando por el ecuador o un paralelo
        rumbo = np.pi/2 if delta_lo > 0 else 3*np.pi/2
        distancia_loxo_nm = abs(np.degrees(delta_lo) * 60 * np.cos(l1))
    else:
        rumbo = np.arctan2(delta_lo, delta_lat_aum)
        distancia_loxo_nm = abs(np.degrees(delta_l) * 60 / np.cos(rumbo))
        
    ahorro = distancia_loxo_nm - distancia_orto_nm
    
    print(f"🌍 Distancia Loxodrómica (Rumbo constante): {distancia_loxo_nm:.1f} NM")
    print(f"🌎 Distancia Ortodrómica (Círculo máximo): {distancia_orto_nm:.1f} NM")
    print(f"\n✅ Ahorro al navegar por círculo máximo: {ahorro:.1f} Millas Náuticas")
    
    if ahorro < 10:
        print("Para travesías cortas o cerca del ecuador, el ahorro es despreciable. Compensa navegar a rumbo constante (Loxodrómica).")
    else:
        print("Para travesías oceánicas largas, especialmente en altas latitudes, la Ortodrómica ahorra mucha distancia, aunque requiere recalcular el rumbo continuamente.")

# Ej: Cadiz (36N, 6W) a La Habana (23N, 82W)
lat1 = widgets.FloatSlider(value=36.0, min=-90, max=90, description='Lat 1 (º):')
lon1 = widgets.FloatSlider(value=-6.0, min=-180, max=180, description='Lon 1 (º):')
lat2 = widgets.FloatSlider(value=23.0, min=-90, max=90, description='Lat 2 (º):')
lon2 = widgets.FloatSlider(value=-82.0, min=-180, max=180, description='Lon 2 (º):')

out = widgets.interactive_output(calcular_distancias, {'lat1_deg': lat1, 'lon1_deg': lon1, 'lat2_deg': lat2, 'lon2_deg': lon2})
display(widgets.VBox([lat1, lon1, lat2, lon2, out]))
